In [4]:
import pathlib
import warnings
import json

import numpy as np
import scipy.sparse as sp
import joblib
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

ROOT      = pathlib.Path("..").resolve()
ARTIFACTS = ROOT / "artifacts"
MODELS    = ROOT / "models"
MODELS.mkdir(exist_ok=True)

RANDOM_STATE  = 42
TEST_SIZE     = 0.20
OPTUNA_TRIALS = 60    # increase for better tuning; 60 is a good balance
N_JOBS        = 1     # set to -1 if RAM > 16 GB, else keep 1 to avoid OOM

print(f"Artifacts : {ARTIFACTS}")
print(f"Models    : {MODELS}")
print(f"Optuna trials : {OPTUNA_TRIALS}")

Artifacts : C:\Users\BIT\APT-Attribution-Engine\artifacts
Models    : C:\Users\BIT\APT-Attribution-Engine\models
Optuna trials : 60


In [6]:
import joblib

X = joblib.load(ARTIFACTS / "X.joblib")
y = joblib.load(ARTIFACTS / "y.joblib")
feature_vocab = joblib.load(ARTIFACTS / "feature_vocab.joblib")

print("X shape:", X.shape)
print("Classes:", len(set(y)))

X shape: (5040, 204)
Classes: 168


In [7]:
from collections import Counter
import numpy as np
from sklearn.model_selection import train_test_split

count_by_class = Counter(y)

single_classes = {c for c, n in count_by_class.items() if n < 2}

multi_mask = np.array([label not in single_classes for label in y])

idx_single = np.where(~multi_mask)[0]
idx_multi = np.where(multi_mask)[0]

X_multi = X[idx_multi]
y_multi = np.array(y)[idx_multi]

X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_multi,
    y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

# Single-sample classes always in train
if len(idx_single) > 0:
    X_train = np.vstack([X[idx_single], X_multi_train])
    y_train = np.concatenate([np.array(y)[idx_single], y_multi_train])
else:
    X_train = X_multi_train
    y_train = y_multi_train

X_test = X_multi_test
y_test = y_multi_test

print("Total classes:", len(count_by_class))
print("Single-sample classes:", len(single_classes))
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Total classes: 168
Single-sample classes: 0
Train shape: (4032, 204)
Test shape: (1008, 204)


In [8]:
from imblearn.over_sampling import SMOTE

smote_k = 2   # conservative k — sparse groups have very few samples

class_counts_train = Counter(y_train)
classes_eligible   = [c for c, n in class_counts_train.items() if n >= smote_k + 1]
classes_skipped    = [c for c, n in class_counts_train.items() if n < smote_k + 1]

print(f"Classes eligible for SMOTE : {len(classes_eligible)}")
print(f"Classes skipped (too few)  : {len(classes_skipped)}")

if len(classes_skipped) > 0:
    skipped_names = classes_skipped[:10]
if len(classes_skipped) > 0:
    print("Sample skipped:", list(classes_skipped[:10]))
else:
    print("No classes skipped")
try:
    smote = SMOTE(k_neighbors=smote_k, random_state=RANDOM_STATE)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    X_train_res = sp.csr_matrix(X_train_res)
    print(f"\nAfter SMOTE → train size: {X_train_res.shape[0]}")
except Exception as e:
    print(f"SMOTE failed: {e}\nFalling back to original training set.")
    X_train_res, y_train_res = X_train, y_train


Classes eligible for SMOTE : 168
Classes skipped (too few)  : 0
No classes skipped

After SMOTE → train size: 4032


In [10]:
def make_xgb(params=None):
    defaults = dict(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbosity=0,
    )
    if params:
        defaults.update(params)
    return XGBClassifier(**defaults)

def make_rf(params=None):
    defaults = dict(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced_subsample",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )
    if params:
        defaults.update(params)
    return RandomForestClassifier(**defaults)


def make_svm():
    base = SVC(
        kernel="linear",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=5000,
    )
    return CalibratedClassifierCV(base, cv=3, method="isotonic")


print("Classifier factories defined:")
print("  make_xgb()  — XGBoost (hist tree, mlogloss)")
print("  make_rf()   — Random Forest (balanced_subsample)")
print("  make_svm()  — CalibratedClassifierCV w/ linear SVC (isotonic)")


Classifier factories defined:
  make_xgb()  — XGBoost (hist tree, mlogloss)
  make_rf()   — Random Forest (balanced_subsample)
  make_svm()  — CalibratedClassifierCV w/ linear SVC (isotonic)


In [11]:
X_opt = X_train_res.toarray() if sp.issparse(X_train_res) else X_train_res
cv    = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    xgb_params = dict(
        n_estimators    = trial.suggest_int("xgb_n_est",   100, 500, step=50),
        max_depth       = trial.suggest_int("xgb_depth",   3, 8),
        learning_rate   = trial.suggest_float("xgb_lr",    0.01, 0.3, log=True),
        subsample       = trial.suggest_float("xgb_sub",   0.5, 1.0),
        colsample_bytree= trial.suggest_float("xgb_col",   0.4, 1.0),
    )
    rf_params = dict(
        n_estimators = trial.suggest_int("rf_n_est", 100, 400, step=50),
        max_depth    = trial.suggest_int("rf_depth", 5, 30),
        min_samples_split = trial.suggest_int("rf_min_split", 2, 10),
    )
    w_xgb = trial.suggest_int("w_xgb", 1, 4)
    w_rf  = trial.suggest_int("w_rf",  1, 3)
    w_svm = 1  # SVM weight fixed — too slow to tune here

    ensemble = VotingClassifier(
        estimators=[
            ("xgb", make_xgb(xgb_params)),
            ("rf",  make_rf(rf_params)),
            ("svm", make_svm()),
        ],
        voting="soft",
        weights=[w_xgb, w_rf, w_svm],
        n_jobs=1,
    )

    scores = cross_val_score(
        ensemble, X_opt, y_train_res,
        cv=cv, scoring="f1_macro", n_jobs=1,
    )
    return scores.mean()

print(f"Starting Optuna search — {OPTUNA_TRIALS} trials (this may take 10–30 min)...")
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

print(f"\nBest macro F1 (CV)  : {study.best_value:.4f}")
print(f"Best params         : {study.best_params}")


Starting Optuna search — 60 trials (this may take 10–30 min)...

Best macro F1 (CV)  : 0.9234
Best params         : {'xgb_n_est': 250, 'xgb_depth': 7, 'xgb_lr': 0.011661300819679149, 'xgb_sub': 0.5556499720505148, 'xgb_col': 0.7980506399839633, 'rf_n_est': 300, 'rf_depth': 29, 'rf_min_split': 2, 'w_xgb': 1, 'w_rf': 3}


In [12]:
bp = study.best_params

final_ensemble = VotingClassifier(
    estimators=[
        ("xgb", make_xgb({
            "n_estimators"    : bp["xgb_n_est"],
            "max_depth"       : bp["xgb_depth"],
            "learning_rate"   : bp["xgb_lr"],
            "subsample"       : bp["xgb_sub"],
            "colsample_bytree": bp["xgb_col"],
        })),
        ("rf", make_rf({
            "n_estimators"    : bp["rf_n_est"],
            "max_depth"       : bp["rf_depth"],
            "min_samples_split": bp["rf_min_split"],
        })),
        ("svm", make_svm()),
    ],
    voting="soft",
    weights=[bp["w_xgb"], bp["w_rf"], 1],
    n_jobs=1,
)

print("Training final ensemble on full resampled training set...")
X_final = X_train_res.toarray() if sp.issparse(X_train_res) else X_train_res
final_ensemble.fit(X_final, y_train_res)
print("Training complete ✓")


Training final ensemble on full resampled training set...
Training complete ✓


In [13]:
X_test_dense = X_test.toarray() if sp.issparse(X_test) else X_test
y_pred = final_ensemble.predict(X_test_dense)

f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)

print(f"Sanity check on held-out test set:")
print(f"  Macro F1 : {f1_macro:.4f}  (target ≥ 0.62)")
print(f"  Micro F1 : {f1_micro:.4f}")

if f1_macro < 0.62:
    print("\n⚠  Below threshold — see Phase 4 notebook for tuning strategies.")
else:
    print("\n✓  Threshold met. Proceed to Phase 4 for full evaluation.")


Sanity check on held-out test set:
  Macro F1 : 0.9431  (target ≥ 0.62)
  Micro F1 : 0.9435

✓  Threshold met. Proceed to Phase 4 for full evaluation.


In [14]:
joblib.dump(final_ensemble, MODELS / "model.joblib")

metadata = {
   "optuna_best_value": 0.9287,
"optuna_best_params": bp,
    "n_trials"          : OPTUNA_TRIALS,
    "smote_k"           : smote_k,
    "test_size"         : TEST_SIZE,
    "random_state"      : RANDOM_STATE,
   "n_classes"  : int(len(set(y))),
"n_features" : int(X.shape[1]),
    "sanity_macro_f1"   : float(f1_macro),
    "sanity_micro_f1"   : float(f1_micro),
}
with open(MODELS / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(f"  models/model.joblib")
print(f"  models/training_metadata.json")
print()
print("=" * 55)
print("  Phase 3 complete — Training Pipeline")
print("=" * 55)
print("  Next → 03_evaluation.ipynb")
print("=" * 55)


Saved:
  models/model.joblib
  models/training_metadata.json

  Phase 3 complete — Training Pipeline
  Next → 03_evaluation.ipynb
